This script demonstrates various strategies to design and judge the results of an experiment.  
We will simulate a promotion to increase sales of a treatment group.

A/B test: Compares the treatment group's outcome with the control group's outcome, typically after random assignment.



DiD: Compares the change in the treatment group with the change in the control group, using before-and-after data.

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import datetime as dt
from sklearn.linear_model import LinearRegression

from pathlib import Path
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

project_dir = Path.cwd()
data_dir = project_dir / 'Raw Data'
results_dict = {}

# Get data

In [3]:
# load single-store sales data
sales_long = pd.read_csv(
    data_dir / 'sales_long.csv',
    parse_dates=['date'],
    low_memory=False
)

# recall the level of this dataset
# i.e. a single store with many items over many dates
sales_long.groupby(['store_id','date','cat_id'])[['item_id']].nunique().head()

item_id
store_id date       cat_id            
CA_1     2011-01-29 FOODS         1437
                    HOBBIES        565
                    HOUSEHOLD     1047
         2011-01-30 FOODS         1437
                    HOBBIES        565

# Summarize

In [4]:
# inspect the date range and number of unique items
print(f"Date range: {sales_long['date'].min()} to {sales_long['date'].max()}")
print(f"Unique items: {sales_long['item_id'].nunique():,}")
print(f"Unique dates: {sales_long['date'].nunique():,}")

# inspect sales coverage by item
item_summary = sales_long.groupby('item_id').agg(
    first_date=('date', 'min'),
    last_date=('date', 'max'),
    days_observed=('date', 'nunique'),
    total_units_sold=('units_sold', 'sum'),
    avg_daily_units=('units_sold', 'mean')
)

display(item_summary.describe())

Date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00
Unique items: 3,049
Unique dates: 1,913


,days_observed,total_units_sold,avg_daily_units
count,3049.0,3049.000000,3049.000000
mean,1913.0,2524.833060,1.319829
std,0.0,5352.040144,2.797721
min,1913.0,21.000000,0.010978
25%,1913.0,491.000000,0.256665
50%,1913.0,1122.000000,0.586513
75%,1913.0,2446.000000,1.278620
max,1913.0,127203.000000,66.493988


# Create control and test groups

In [5]:
# identify each item's category and average daily sales
item_summary = (
    sales_long.groupby(['item_id', 'cat_id'], as_index=False)
    .agg(avg_daily_units=('units_sold', 'mean'))
)

# set a reproducible random seed
np.random.seed(42)

# randomly assign items to treatment and control within each category
item_summary['group'] = (
    item_summary.groupby('cat_id')['item_id']
    .transform(lambda items: np.random.permutation(
        ['treatment'] * (len(items) // 2)
        + ['control'] * (len(items) - len(items) // 2)
    ))
)

# inspect group sizes and baseline sales
display(
    item_summary.groupby('group').agg(
        item_count=('item_id', 'nunique'),
        avg_daily_units=('avg_daily_units', 'mean'),
        median_daily_units=('avg_daily_units', 'median')
    )
)

# inspect category balance
display(pd.crosstab(item_summary['cat_id'], item_summary['group']))

,item_count,avg_daily_units,median_daily_units
group,,,
control,1526,1.321500,0.586513
treatment,1523,1.318155,0.586513


group,control,treatment
cat_id,,
FOODS,719,718
HOBBIES,283,282
HOUSEHOLD,524,523


# Simulate the promotion
For each existing unit sold within the treatment group, generate a 10% chance of increasing by one additional unit.

In [6]:
# define the experiment period
# use the final 28 days of historical data as the simulated experiment window
# all earlier dates will serve as the pre-experiment period
experiment_days = 28
experiment_start = sales_long['date'].max() - pd.Timedelta(days=experiment_days - 1)

# create the experimental dataset
# attach each item's randomized treatment/control assignment to its daily sales
# each row still represents one item on one date
experiment_data = sales_long.merge(
    item_summary[['item_id', 'group']], # item_summary contains label for control/test group
    on='item_id',
    how='left'
).copy()

# flag observations before and during the experiment
# post = 0 for the historical baseline period
# post = 1 for dates within the 28-day experiment window
experiment_data['post'] = (
    experiment_data['date'] >= experiment_start
).astype(int)

# preserve observed sales as the baseline outcome
# these are the original sales values from the dataset
# keeping them lets us compare actual historical sales with simulated outcomes
experiment_data['observed_units'] = experiment_data['units_sold']

# initialize simulated sales with the original observed sales
# control items and all pre-experiment observations remain unchanged
experiment_data['simulated_units'] = experiment_data['observed_units']

# identify treatment items during the experiment period
# only these rows will receive the simulated promotional effect
treatment_mask = (
    (experiment_data['group'] == 'treatment')
    & (experiment_data['post'] == 1)
)

# simulate incremental units sold from the promotion
# for each existing unit sold, generate a 10% chance of one additional unit
# this uses a binomial distribution, which models successes across repeated trials
# the generated extra units are added to the original sales
# this creates a synthetic treatment effect rather than a real measured effect
experiment_data.loc[treatment_mask, 'simulated_units'] = (
    experiment_data.loc[treatment_mask, 'observed_units']
    + np.random.binomial(
        n=experiment_data.loc[treatment_mask, 'observed_units'].astype(int),
        p=0.10
    )
)

# inspect the experiment setup
# verify the dates and number of treatment observations affected
print(f"Experiment start: {experiment_start.date()}")
print(f"Experiment end: {sales_long['date'].max().date()}")
print(f"Experiment duration: {experiment_days} days")
print(f"Simulated treatment observations: {treatment_mask.sum():,}")

# compare average daily units before and during the experiment
# observed_units shows the original sales
# simulated_units includes the added promotional units for treatment items
display(
    experiment_data.groupby(['group', 'post'])[
        ['observed_units', 'simulated_units']
    ].mean()
)

Experiment start: 2016-03-28
Experiment end: 2016-04-24
Experiment duration: 28 days
Simulated treatment observations: 42,644


observed_units  simulated_units
group     post                                 
control   0           1.319269         1.319269
          1           1.471728         1.471728
treatment 0           1.315069         1.315069
          1           1.525912         1.677540

# Collect experiment results

In [7]:
# isolate observations from the simulated experiment period
post_experiment = experiment_data.loc[
    experiment_data['post'] == 1
].copy()

# calculate average daily sales for each item during the experiment
# each item contributes one outcome, regardless of how many daily rows it has
item_results = (
    post_experiment.groupby(['item_id', 'cat_id', 'group'], as_index=False)
    .agg(
        observed_avg_units=('observed_units', 'mean'),
        simulated_avg_units=('simulated_units', 'mean')
    )
)

# compare average simulated sales between treatment and control items
# the simulated outcome includes the promotion effect for treatment items
display(
    item_results.groupby('group').agg(
        item_count=('item_id', 'nunique'),
        avg_daily_units=('simulated_avg_units', 'mean'),
        median_daily_units=('simulated_avg_units', 'median')
    )
)

,item_count,avg_daily_units,median_daily_units
group,,,
control,1526,1.471728,0.750000
treatment,1523,1.677540,0.821429


## A/B test

### Lift

In [37]:
# calculate the absolute and percentage difference in average daily sales
group_means = item_results.groupby('group')['simulated_avg_units'].mean()

control_mean = group_means['control']
treatment_mean = group_means['treatment']

# lift (often called incremental lift) is a metric that measures the direct,
# causal improvement or change in an outcome caused by a specific intervention, 
# treatment, or campaign compared to a baseline
absolute_lift = treatment_mean - control_mean
percentage_lift = absolute_lift / control_mean * 100

print(f"Control average daily units:   {control_mean:.4f}")
print(f"Treatment average daily units: {treatment_mean:.4f}")
print(f"Absolute lift:                 {absolute_lift:.4f} units per item per day")
print(f"Percentage lift:               {percentage_lift:.2f}%")

print("\nInterpretation: Say there are 5 items in the treatment group.\
\nIf I run the promotion for 100 days, and I look at those same five items,\
\nI'll expect to see roughly 20 additional units per item.")

results_dict['A/B'] = {
    'control_mean': control_mean,
    'treatment_mean': treatment_mean,
    'absolute_lift': absolute_lift,
    'percentage_lift': percentage_lift,
}

Control average daily units:   1.4717
Treatment average daily units: 1.6775
Absolute lift:                 0.2058 units per item per day
Percentage lift:               13.98%

Interpretation: Say there are 5 items in the treatment group.
If I run the promotion for 100 days, and I look at those same five items,
I'll expect to see roughly 20 additional units per item.


### Statistical significance

In [28]:
from scipy import stats

# calculate the standard error of the difference in means
# sample variance measures how spread out sales are within each group
treatment_values = item_results.loc[
    item_results['group'] == 'treatment',
    'simulated_avg_units'
]

control_values = item_results.loc[
    item_results['group'] == 'control',
    'simulated_avg_units'
]

n_treatment = len(treatment_values)
n_control = len(control_values)

var_treatment = treatment_values.var(ddof=1)
var_control = control_values.var(ddof=1)

# estimates how much uncertainty there is in the measured promotion lift
standard_error = np.sqrt(
    var_treatment / n_treatment
    + var_control / n_control
)

# calculate the Welch-Satterthwaite degrees of freedom
# this accounts for different variances and sample sizes
degrees_freedom = (
    (var_treatment / n_treatment + var_control / n_control) ** 2
    / (
        (var_treatment / n_treatment) ** 2 / (n_treatment - 1)
        + (var_control / n_control) ** 2 / (n_control - 1)
    )
)

# calculate the t-statistic
# measures how many standard errors the estimated lift is from zero
t_statistic = absolute_lift / standard_error

# calculate the critical t-value for a 95% confidence interval
critical_value = stats.t.ppf(0.975, df=degrees_freedom)

# calculate the confidence interval for the absolute lift
ci_lower = absolute_lift - critical_value * standard_error
ci_upper = absolute_lift + critical_value * standard_error

# calculate the two-sided p-value from the t-distribution
p_value = 2 * stats.t.sf(abs(t_statistic), df=degrees_freedom)

# display the complete analysis
print(f"Standard error:               {standard_error:.4f}")
print(f"Degrees of freedom:            {degrees_freedom:.2f}")
print(f"T-statistic:                   {t_statistic:.4f}")
print(f"95% confidence interval:       ({ci_lower:.4f}, {ci_upper:.4f})")
print(f"P-value:                      {p_value:.6g}")

# interpret statistical significance using a 5% threshold
if p_value < 0.05:
    print("The difference is statistically significant at alpha = 0.05.")
else:
    print("The difference is not statistically significant at alpha = 0.05.")
    
results_dict['A/B_inference'] = {
    'n_treatment': n_treatment,
    'n_control': n_control,
    'var_treatment': var_treatment,
    'var_control': var_control,
    'standard_error': standard_error,
    'degrees_freedom': degrees_freedom,
    't_statistic': t_statistic,
    'critical_value': critical_value,
    'ci_lower': ci_lower,
    'ci_upper': ci_upper,
    'p_value': p_value,
    'statistically_significant': p_value < 0.05,
}

Standard error:               0.1059
Degrees of freedom:            3021.30
T-statistic:                   1.9442
95% confidence interval:       (-0.0018, 0.4134)
P-value:                      0.0519627
The difference is not statistically significant at alpha = 0.05.


## A/B conclusion
The treatment group showed a positive lift of approximately 13.98% in average daily units sold. However, the difference was not statistically significant at the 5% level (p = 0.052). The 95% confidence interval for the absolute lift included zero, so we cannot conclude that the promotion increased sales beyond what could plausibly be explained by random variation.


However, at p = 0.0519627, we are very close to statistical significance at the alpha = 0.05 level.

# DiD (Difference-in-Differences)
A quasi-experimental research design used to estimate the causal effect of an intervention, policy, or treatment by comparing the change in outcomes over time between a treated group and a control group.

For DiD, check the parallel-trends assumption.  
Before the promotion, the treatment and comparison groups should have followed reasonably similar sales trends.

### Regression

In [29]:
import statsmodels.formula.api as smf

# Calculate each item's average daily sales before and during the experiment.
# Each item contributes one average for each period.
item_period_results = (
    experiment_data
    .groupby(['item_id', 'cat_id', 'group', 'post'], as_index=False)
    .agg(
        avg_daily_units=('simulated_units', 'mean')
    )
)

# Run Difference-in-Differences (DiD) regression.
# C(item_id) controls for each item's baseline sales level.
# post measures the overall change from before to during the experiment.
# group_treatment:post estimates the additional change for treatment items.
did_model = smf.ols(
    'avg_daily_units ~ C(item_id) + post + C(group):post',
    data=item_period_results
).fit(
    cov_type='cluster',
    cov_kwds={'groups': item_period_results['item_id']}
)

# Extract the treatment group's additional change.
did_effect = did_model.params['C(group)[T.treatment]:post']
did_ci = did_model.conf_int().loc['C(group)[T.treatment]:post']
did_p_value = did_model.pvalues['C(group)[T.treatment]:post']

# Display the DiD estimate and uncertainty.
print(f"DiD estimated lift:            {did_effect:.4f} units per item per day")
print(f"95% confidence interval:       ({did_ci.iloc[0]:.4f}, {did_ci.iloc[1]:.4f})")
print(f"P-value:                       {did_p_value:.6g}")

if did_p_value < 0.05:
    print("The DiD estimate is statistically significant at alpha = 0.05.")
else:
    print("The DiD estimate is not statistically significant at alpha = 0.05.")
    
results_dict['DiD_regression'] = {
    'did_effect': did_effect,
    'ci_lower': did_ci.iloc[0],
    'ci_upper': did_ci.iloc[1],
    'p_value': did_p_value,
    'statistically_significant': did_p_value < 0.05,
}

DiD estimated lift:            0.2100 units per item per day
95% confidence interval:       (0.0525, 0.3675)
P-value:                       0.00895187
The DiD estimate is statistically significant at alpha = 0.05.


### Arithmetic

In [30]:

# Calculate each item's average daily sales before and during the experiment.
item_period_results = (
    experiment_data
    .groupby(['item_id', 'cat_id', 'group', 'post'], as_index=False)
    .agg(avg_daily_units=('simulated_units', 'mean'))
)

# Calculate the average sales for each group in each period.
group_period_means = (
    item_period_results
    .groupby(['group', 'post'])['avg_daily_units']
    .mean()
    .unstack()
)

# Calculate the change in average sales for each group.
# post = 0 represents the period before the promotion.
# post = 1 represents the period during the promotion.
control_change = (
    group_period_means.loc['control', 1]
    - group_period_means.loc['control', 0]
)

treatment_change = (
    group_period_means.loc['treatment', 1]
    - group_period_means.loc['treatment', 0]
)

# Subtract the control group's change from the treatment group's change.
# This is the Difference-in-Differences estimate.
did_effect = treatment_change - control_change

# Display the group averages and changes.
print("Average daily units per item:")
print(group_period_means)

print(f"\nControl group change:          {control_change:.4f}")
print(f"Treatment group change:        {treatment_change:.4f}")
print(f"DiD estimated lift:            {did_effect:.4f} units per item per day")

results_dict['DiD_arithmetic'] = {
    'control_change': control_change,
    'treatment_change': treatment_change,
    'did_effect': did_effect,
    'control_mean_before': group_period_means.loc['control', 0],
    'control_mean_after': group_period_means.loc['control', 1],
    'treatment_mean_before': group_period_means.loc['treatment', 0],
    'treatment_mean_after': group_period_means.loc['treatment', 1],
}

Average daily units per item:
post              0         1
group                        
control    1.319269  1.471728
treatment  1.315069  1.677540

Control group change:          0.1525
Treatment group change:        0.3625
DiD estimated lift:            0.2100 units per item per day


# Conclusion
All three approaches estimated a positive lift of about 0.21 units per item per day. The conventional A/B comparison narrowly missed statistical significance, while the DiD regression produced a more precise estimate by accounting for item-specific baseline differences and changes over time. I would investigate why the uncertainty differs and verify the model assumptions before concluding that DiD provides stronger evidence.

In [39]:
summary_df = pd.DataFrame({
    'A/B': {
        'Estimated lift': results_dict['A/B']['absolute_lift'],
        'Percentage lift (%)': results_dict['A/B']['percentage_lift'],
        '95% CI lower': results_dict['A/B_inference']['ci_lower'],
        '95% CI upper': results_dict['A/B_inference']['ci_upper'],
        'P-value': results_dict['A/B_inference']['p_value'],
        'Statistically significant': results_dict['A/B_inference']['statistically_significant'],
    },
    'DiD regression': {
        'Estimated lift': results_dict['DiD_regression']['did_effect'],
        'Percentage lift (%)': None,
        '95% CI lower': results_dict['DiD_regression']['ci_lower'],
        '95% CI upper': results_dict['DiD_regression']['ci_upper'],
        'P-value': results_dict['DiD_regression']['p_value'],
        'Statistically significant': results_dict['DiD_regression']['statistically_significant'],
    },
    'DiD arithmetic': {
        'Estimated lift': results_dict['DiD_arithmetic']['did_effect'],
        'Percentage lift (%)': None,
        '95% CI lower': None,
        '95% CI upper': None,
        'P-value': None,
        'Statistically significant': None,
    },
}).T

display(summary_df.round(4))

,Estimated lift,Percentage lift (%),95% CI lower,95% CI upper,P-value,Statistically significant
A/B,0.205811,13.984342,-0.001751,0.413374,0.051963,False
DiD regression,0.210012,None,0.052539,0.367484,0.008952,True
DiD arithmetic,0.210012,NaN,NaN,NaN,NaN,NaN


In [40]:
print('done')

done
